In [1]:
"""
01b_merge_upstream_tss.ipynb
-----------------------------
Merges upstream TSS time series for all stations using area-weighted sums.
For each variable and each station, the merged time series represents
the full upstream catchment (station + all upstream sub-basins).

Note: q (discharge) is already accumulated → skipped.

Output per variable:
    {var}_full_upstream.parquet  → (date × station_id) full period
    {var}_full_upstream_summary.csv → statistics per station
"""

import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict



In [2]:
# =============================================================================
# TSS READER
# =============================================================================

def read_tss_single_gauge(tss_path: Path, start_date: str) -> pd.Series:
    if not tss_path.exists():
        return pd.Series(dtype=float)
    with open(tss_path) as f:
        lines = f.readlines()
    try:
        n_scalars = int(lines[1].strip())
    except (IndexError, ValueError):
        n_scalars = 1
    skiprows = 2 + n_scalars
    try:
        df = pd.read_csv(
            tss_path, skiprows=skiprows, sep=r'\s+',
            header=None, usecols=[0, 1],
            dtype={0: float, 1: float},
        )
        dates  = pd.date_range(start=start_date, periods=len(df), freq="D")
        series = pd.Series(df.iloc[:, 1].values, index=dates, dtype=float)
        series = series.replace(-9999.0, np.nan)
        series = series.where(series > -9000)
        return series
    except Exception as e:
        print(f"    ⚠️  Could not parse {tss_path.name}: {e}")
        return pd.Series(dtype=float)

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================

BASE_FILE  = Path.cwd() / "../glofas5_hydrobot.csv"
DIR_TSS    = Path("/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/long_term_runs/01_Hydrology/")
DIR_GRAPHS = Path.cwd() / "../results"
DIR_OUT    = Path.cwd() / "../results/merged_tss"
DIR_OUT.mkdir(parents=True, exist_ok=True)

RUN_NAME   = "long_term_run"
START_DATE = "1975-01-02"
FULL_START = "1991-01-01"
FULL_END   = "2020-12-31"

# Variables to merge (q not skipped due to organizational/structural reasons though already accumulated)
VARS_MERGE = {
    "rain":        "rainUps",
    "ewact":       "evaopen",
    "eta":         "actEvapo",
    "es":          "esActUps",
    "transp":      "tActUps",
    "intcpt":      "ewIntActUps",
    "swe":         "scovUps",
    "sf":          "snowUps",
    "smlt":        "snowMeltUps",
    "total_runoff":"totalRunoffUps",
    "perc":        "percUZLZUps",
    "qb_up":       "qUzUps",
    "qb_low":      "qLzUps",
    "gwloss":      "gwLossUps",
    "lz":          "lzUps",
    "uz":          "uzUps",
    "theta":       "theta1total",
    "theta2":      "theta2total",
    "theta3":      "theta3total",
    "q":           "dis"
}

In [4]:
# =============================================================================
# LOAD BASE INFO + GRAPHS
# =============================================================================

print("Loading base info...")
glofas5_base_info = pd.read_csv(BASE_FILE)
glofas5_base_info["ID"] = glofas5_base_info["ID"].astype(int)
area_lookup = glofas5_base_info.set_index("ID")["DrainageArea_LDD"]

print("Loading connectivity graphs...")
with open(DIR_GRAPHS / "downstream_graph.json") as f:
    downstream = {int(float(k)): (int(float(v)) if v is not None else None)
                  for k, v in json.load(f).items()}

with open(DIR_GRAPHS / "upstream_lookup.json") as f:
    upstream_lookup = {int(float(k)): [int(float(v)) for v in vs]
                       for k, vs in json.load(f).items()}

print(f"  Downstream graph: {len(downstream)} stations")
print(f"  Upstream lookup:  {len(upstream_lookup)} stations")

# =============================================================================
# COMPUTE SUB-BASIN AREAS
# =============================================================================

print("\nComputing sub-basin areas...")
area_sub = {}
for sid in downstream.keys():
    # Direct upstream neighbors
    direct_upstream = [k for k, v in downstream.items() if v == sid]
    area_total      = area_lookup.get(sid, np.nan)
    area_up_sum     = sum(area_lookup.get(u, 0) for u in direct_upstream)
    area_sub[sid]   = max(area_total - area_up_sum, 0)  # clip to 0

# Sanity check
area_sub_df = pd.Series(area_sub)
print(f"  Sub-basin areas — min: {area_sub_df.min():.1f}, "
      f"max: {area_sub_df.max():.1f}, "
      f"mean: {area_sub_df.mean():.1f} km²")
print(f"  Negative areas clipped to 0: {(area_sub_df <= 0).sum()} stations")

Loading base info...
Loading connectivity graphs...
  Downstream graph: 5379 stations
  Upstream lookup:  5379 stations

Computing sub-basin areas...
  Sub-basin areas — min: 109.1, max: 3070786.2, mean: 13034.7 km²
  Negative areas clipped to 0: 0 stations


In [5]:

# =============================================================================
# DISCOVER TSS FOLDERS
# =============================================================================

print("\nScanning TSS directory...")
station_folders = {}
for tss_file in DIR_TSS.rglob(f"dis{RUN_NAME}.tss"):
    folder = tss_file.parent
    try:
        sid = int(folder.parts[-3])
        if sid in downstream:
            station_folders[sid] = folder
    except (ValueError, IndexError):
        continue

print(f"  Found TSS folders for {len(station_folders)} / {len(downstream)} stations")


Scanning TSS directory...
  Found TSS folders for 5262 / 5379 stations


In [6]:
# =============================================================================
# MAIN LOOP — per variable
# =============================================================================
date_index = pd.date_range(start=FULL_START, end=FULL_END, freq="D")

# Loop over Variables
for varname, tss_prefix in VARS_MERGE.items():
    print(f"\n{'='*60}")
    print(f"Processing variable: {varname} ({tss_prefix})")
    print(f"{'='*60}")

    result  = {}
    skipped = []

    # Loop over Stations
    for sid in tqdm(sorted(station_folders.keys()), desc=f"{varname}"):

        all_ids       = [sid] + upstream_lookup.get(sid, [])
        available_ids = [i for i in all_ids if i in station_folders]

        if not available_ids:
            skipped.append(sid)
            result[sid] = pd.Series(np.nan, index=date_index)
            continue

        weighted_sum = pd.Series(0.0, index=date_index)
        total_weight = pd.Series(0.0, index=date_index)  # per-day
        # Loop over upstream stations
        for uid in available_ids:
            # q braucht kein weighted sum — nur Station selbst
            if varname == "q" and uid != sid:
                continue
            tss_path = station_folders[uid] / f"{tss_prefix}{RUN_NAME}.tss"
            ts = read_tss_single_gauge(tss_path, START_DATE)
            if ts.empty:
                continue

            ts = ts.reindex(date_index)

            w = area_sub.get(uid, 0)
            if w <= 0:
                w = area_lookup.get(uid, 1)

            valid_mask    = ts.notna()
            weighted_sum += ts.fillna(0) * w
            total_weight += valid_mask * w  # nur wo ts nicht NaN

        mask           = total_weight > 0
        result[sid]    = (weighted_sum / total_weight).where(mask, np.nan)

    # ── Build DataFrame ──
    df_out            = pd.DataFrame(result, index=date_index)
    df_out.index.name = "date"
    df_out.columns    = df_out.columns.astype(str)

    print(f"  Shape:   {df_out.shape}")
    print(f"  Skipped: {len(skipped)} stations")

    # ── Save Parquet ──
    parquet_path = DIR_OUT / f"{varname}_full_upstream.parquet"
    df_out.to_parquet(parquet_path)
    print(f"  Parquet → {parquet_path}")

    # ── Save Summary CSV ──
    summary            = df_out.agg(["mean", "std", "min", "max"]).T
    summary.index.name = "station_id"
    summary["n_valid"] = df_out.notna().sum()
    csv_path           = DIR_OUT / f"{varname}_full_upstream_summary.csv"
    summary.to_csv(csv_path)
    print(f"  Summary CSV → {csv_path}")

print("\nAll variables processed!")


Processing variable: rain (rainUps)


rain: 100%|██████████| 5262/5262 [06:56<00:00, 12.63it/s]


  Shape:   (10958, 5262)
  Skipped: 0 stations
  Parquet → /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/scripts/../results/merged_tss/rain_full_upstream.parquet
  Summary CSV → /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/scripts/../results/merged_tss/rain_full_upstream_summary.csv

Processing variable: ewact (evaopen)


ewact: 100%|██████████| 5262/5262 [00:14<00:00, 365.97it/s] 


  Shape:   (10958, 5262)
  Skipped: 0 stations
  Parquet → /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/scripts/../results/merged_tss/ewact_full_upstream.parquet
  Summary CSV → /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/scripts/../results/merged_tss/ewact_full_upstream_summary.csv

Processing variable: eta (actEvapo)


...

  Shape:   (10958, 5262)
  Skipped: 0 stations
  Parquet → /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/scripts/../results/merged_tss/theta_full_upstream.parquet
  Summary CSV → /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/scripts/../results/merged_tss/theta_full_upstream_summary.csv

Processing variable: theta2 (theta2total)


theta2: 100%|██████████| 5262/5262 [06:53<00:00, 12.71it/s]  


  Shape:   (10958, 5262)
  Skipped: 0 stations
  Parquet → /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/scripts/../results/merged_tss/theta2_full_upstream.parquet
  Summary CSV → /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/scripts/../results/merged_tss/theta2_full_upstream_summary.csv

Processing variable: theta3 (theta3total)


theta3: 100%|██████████| 5262/5262 [07:17<00:00, 12.03it/s]  


  Shape:   (10958, 5262)
  Skipped: 0 stations
  Parquet → /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/scripts/../results/merged_tss/theta3_full_upstream.parquet
  Summary CSV → /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/scripts/../results/merged_tss/theta3_full_upstream_summary.csv

All variables processed!
